In [1]:
# ====================================================================
# Developer Utility: Module Auto-Reloading
# --------------------------------------------------------------------
# Uncomment the lines below if you are actively modifying the underlying 
# pi-metaboqc source code. It ensures that changes in .py files are 
# dynamically reloaded without restarting the Jupyter kernel.
# ====================================================================

%load_ext autoreload
%autoreload 2

# ${\pi}$-metaboqc: Interactive Analytical Workflow

This notebook provides an interactive, step-by-step execution orchestration of the `pi-metaboqc` metabolomics data quality control (QC) pipeline. 

By executing each cell sequentially, you can trace the data provenance, inspect intermediate matrices, and visualize quality assessment (QA) diagnostics at each stage of the computational framework.

## Step 00: Environment Initialization
This phase initializes the `pi-metaboqc` computational environment and performs baseline hardware diagnostics. It ensures that the output directory structure is securely mounted before commencing heavy matrix operations.

In [2]:
import os
import pandas as pd
from loguru import logger

import pimqc
import pimqc.io_utils as iu
import pimqc.report_utils as ru
pimqc.init(check_hardware=True, log_level="DEBUG", show_progress=True)

from pimqc import (
    build_dataset, MetaboIntAssessor, MetaboIntFilter, MetaboIntCorrector,
    MetaboIntImputer, MetaboIntNormalizer
)

# Define standard directories
DATA_DIR = os.path.join("..", "src", "pimqc", "data")
OUTPUT_DIR = os.path.join(".", "tutorial_output")
iu._check_dir_exists(dir_path=OUTPUT_DIR, handle="makedirs")

# Load pipeline parameters
PARAMS_PATH = os.path.join(DATA_DIR, "pipeline_parameters.toml")
params = iu.load_pipeline_config(config_path=PARAMS_PATH)

# Load raw matrices
meta_df = pd.read_csv(
    os.path.join(DATA_DIR, "project_meta.csv"), header=[0]) 

int_df = pd.read_csv(
    os.path.join(DATA_DIR, "project_intensity.csv"), index_col=[0], header=[0])

2026-05-19 13:16:32.113 | DEBUG    | pimqc:_safe_popen:149 - Functional subprocess: 'd:\miniconda3\envs\metaboqc\python.exe' 'd:\miniconda3\envs\metaboqc\Lib\site-packages\cpuinfo\cpuinfo.py' --json
2026-05-19 13:16:33.690 | INFO     | io_utils:print_hardware_diagnostics:120 - 
 🖥️  System Hardware & Power Diagnostics
OS Platform     : Windows 11 (AMD64)
CPU Model       : AMD Ryzen 9 8940HX with Radeon Graphics
Logical Cores   : 32
Physical Cores  : 16
Total RAM       : 31.80 GB
Power Source    : Plugged In (AC) [80% remaining]
Python Version  : 3.13.0
2026-05-19 13:16:33.691 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output.
2026-05-19 13:16:33.697 | SUCCESS  | io_utils:load_pipeline_config:332 - Pipeline configuration successfully loaded and validated via Pydantic.


## Step 01: Raw Dataset Construction
The workflow begins by transforming fragmented raw peak tables and metadata into a standardized `MetaboInt` object. This phase ensures precise coordinate alignment between sample identifiers and feature intensities, establishing a robust structural foundation.

In [3]:
logger.info("Step 01: Dataset Construction...")

step1_dir = os.path.join(OUTPUT_DIR, "01_Raw_Data")
raw_data = build_dataset(
    meta_info=meta_df,
    int_df=int_df,
    pipeline_params=params,
    output_dir=step1_dir
)
is_multi_batch_flag = raw_data.attrs["is_multi_batch"]

2026-05-19 13:16:33.830 | INFO     | __main__:<module>:1 - Step 01: Dataset Construction...
2026-05-19 13:16:33.839 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\01_Raw_Data.
2026-05-19 13:16:33.936 | INFO     | dataset_builder:execute_build:288 - MetaboInt raw dataset saved as: .\tutorial_output\01_Raw_Data\Raw_Data_Intensity.csv
2026-05-19 13:16:33.936 | INFO     | dataset_builder:execute_build:304 - MetaboInt object built: 376 metabolites, 466 samples.


2026-05-19 13:16:34.886 | INFO     | dataset_builder:execute_build:318 - Global acquisition overview plot saved as: .\tutorial_output\01_Raw_Data\Global_Acquisition_Overview.svg
2026-05-19 13:16:34.897 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "build_dataset": 00:00:01.065.


### QA-Step 01: Quality Assessment of Raw Data
Evaluates the baseline data distribution, acquisition sequences, and pooled QC clustering before any computational manipulation occurs.

In [4]:
logger.info("QA-Step 01: Quality Assessment of Raw Data...")

qa_step1_dir = os.path.join(OUTPUT_DIR, "QA_01_Raw_Data")
qa_raw_engine = MetaboIntAssessor(data=raw_data, pipeline_params=params)
qa_raw_engine.execute_assessment(output_dir=qa_step1_dir)

2026-05-19 13:16:34.933 | INFO     | __main__:<module>:1 - QA-Step 01: Quality Assessment of Raw Data...
2026-05-19 13:16:34.934 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_01_Raw_Data.


2026-05-19 13:16:41.066 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_01_Raw_Data\QA_Summary_Dashboard.svg
2026-05-19 13:16:41.066 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:16:41.067 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.133.


## Step 02: Missing Value Classification & Filtering
Implements a topological classification algorithm to segregate missing values into Missing at Random (MAR) and Missing Not at Random (MNAR) based on biological groupings and QC thresholds, strictly filtering out unsalvageable features.

In [5]:
logger.info("Step 02: High Missing Value Feature Filter...")

step2_dir = os.path.join(OUTPUT_DIR, "02_MV_Filtered")
fltr_mv_engine = MetaboIntFilter(data=raw_data, pipeline_params=params)
mv_filter_data = fltr_mv_engine.execute_mv_filtering(output_dir=step2_dir)

2026-05-19 13:16:41.109 | INFO     | __main__:<module>:1 - Step 02: High Missing Value Feature Filter...


2026-05-19 13:16:41.122 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\02_MV_Filtered.


2026-05-19 13:16:42.603 | INFO     | filtering:_execute_s1_visualization:426 - High-MV Filter summary dashboard saved as: .\tutorial_output\02_MV_Filtered\MV_Classification_Dashboard.svg
2026-05-19 13:16:42.604 | SUCCESS  | filtering:execute_mv_filtering:331 - High-missing value feature filtering completed.
2026-05-19 13:16:42.604 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_mv_filtering": 00:00:01.493.


### QA-Step 02: Quality Assessment of High-MV Filtered Data
Evaluates whether global data distributions remain undisturbed after the removal of high-missing-rate features, ensuring no artificial bias is introduced during topological pruning.

In [6]:
logger.info("QA-Step 02: Quality Assessment of High-MV Filtered Data...")

qa_step2_dir = os.path.join(OUTPUT_DIR,  "QA_02_MV_Filtered")
qa_mv_filter_engine = MetaboIntAssessor(data=mv_filter_data, pipeline_params=params)
qa_mv_filter_engine.execute_assessment(output_dir=qa_step2_dir)

2026-05-19 13:16:42.640 | INFO     | __main__:<module>:1 - QA-Step 02: Quality Assessment of High-MV Filtered Data...
2026-05-19 13:16:42.642 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_02_MV_Filtered.


2026-05-19 13:16:49.274 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_02_MV_Filtered\QA_Summary_Dashboard.svg
2026-05-19 13:16:49.275 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:16:49.276 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.634.


## Step 03 & 04: Signal Drift and Batch Effect Mitigation
Utilizes robust machine learning algorithms (e.g., QC-SVR) to model and mitigate systemic technical variations. This step handles both intra-batch instrument signal decay and inter-batch baseline shifts, aligning analytical runs to a unified intensity scale.

In [7]:
logger.info("Step 03 & 04: Signal Drift & Batch Effect Correction...")

step3_4_dir = os.path.join(OUTPUT_DIR, "03_04_Corrected_Data")
sc_engine = MetaboIntCorrector(data=mv_filter_data, pipeline_params=params)
intra_sc_data, inter_sc_data = sc_engine.execute_signal_correction(
    output_dir=step3_4_dir)

2026-05-19 13:16:49.320 | INFO     | __main__:<module>:1 - Step 03 & 04: Signal Drift & Batch Effect Correction...
2026-05-19 13:16:49.321 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\03_04_Corrected_Data.
2026-05-19 13:16:49.454 | INFO     | correction:_calculate_predicted_matrix:392 - Using Joblib Parallel QC-SVR for batch: B1


SC [B1]: 100%|███████████████████████████████████████████████████████████████████| 347/347 [Elapsed: 00:03 | ETA: 00:00]


2026-05-19 13:16:57.274 | INFO     | correction:_calculate_predicted_matrix:392 - Using Joblib Parallel QC-SVR for batch: B2


SC [B2]: 100%|███████████████████████████████████████████████████████████████████| 347/347 [Elapsed: 00:00 | ETA: 00:00]


2026-05-19 13:17:00.789 | INFO     | correction:_calculate_predicted_matrix:392 - Using Joblib Parallel QC-SVR for batch: B3


SC [B3]: 100%|███████████████████████████████████████████████████████████████████| 347/347 [Elapsed: 00:00 | ETA: 00:00]


2026-05-19 13:17:04.254 | INFO     | correction:execute_signal_correction:468 - Intra-correction completed, saved as : .\tutorial_output\03_04_Corrected_Data\Intra_Batch_Corrected_QC-SVR.csv
2026-05-19 13:17:04.616 | INFO     | correction:execute_signal_correction:490 - Inter-correction completed, saved as : .\tutorial_output\03_04_Corrected_Data\Inter_Batch_Corrected_QC-SVR.csv
2026-05-19 13:17:08.568 | SUCCESS  | correction:execute_signal_correction:547 - Data signal drift and batch-effect correction completed.
2026-05-19 13:17:08.570 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_signal_correction": 00:00:19.248.


### QA-Step 03: Quality Assessment of Intra-batch Corrected Data
This stage performs a high-resolution visual and statistical audit to validate the mitigation of within-batch instrument drift. It specifically evaluates whether individual signal trajectories have been successfully normalized to a stable, horizontal baseline while preserving true biological variance.

In [8]:
logger.info("QA-Step 03: Quality Assessment of Intra-batch Corrected Data...")

qa_step3_dir = os.path.join(OUTPUT_DIR, "QA_03_Intra_Corrected_Data")
qa_intra_engine = MetaboIntAssessor(data=intra_sc_data, pipeline_params=params) 
qa_intra_engine.execute_assessment(output_dir=qa_step3_dir)

2026-05-19 13:17:08.606 | INFO     | __main__:<module>:1 - QA-Step 03: Quality Assessment of Intra-batch Corrected Data...
2026-05-19 13:17:08.607 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_03_Intra_Corrected_Data.


2026-05-19 13:17:15.352 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_03_Intra_Corrected_Data\QA_Summary_Dashboard.svg
2026-05-19 13:17:15.353 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:17:15.354 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.746.


### QA-Step 04: Quality Assessment of Inter-batch Corrected Data
This phase analyzes the intensity scale alignment across multiple independent acquisition batches. The assessment focuses on the convergence of Pooled QC clusters in PCA space and the harmonization of median intensity levels, ensuring the dataset is systemically unified for downstream meta-analysis.

In [9]:
logger.info("QA-Step 04: Quality Assessment of Inter-batch Corrected Data...")

qa_step4_dir = os.path.join(OUTPUT_DIR, "QA_04_Inter_Corrected_Data")
qa_inter_engine = MetaboIntAssessor(data=inter_sc_data, pipeline_params=params)
qa_inter_engine.execute_assessment(output_dir=qa_step4_dir)

2026-05-19 13:17:15.399 | INFO     | __main__:<module>:1 - QA-Step 04: Quality Assessment of Inter-batch Corrected Data...
2026-05-19 13:17:15.400 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_04_Inter_Corrected_Data.


2026-05-19 13:17:22.320 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_04_Inter_Corrected_Data\QA_Summary_Dashboard.svg
2026-05-19 13:17:22.321 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:17:22.322 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.921.


## Step 05: Low-Quality Feature Filtering
Performs a rigorous reproducibility check using the analytical variance (RSD) of Pooled QC samples and Blank/QC ratio. Features exhibiting unacceptable technical variance post-correction are permanently eliminated from the quantitative matrix.

In [10]:
logger.info("Step 05: Low-Quality Feature Filtering...")

step5_dir = os.path.join(OUTPUT_DIR, "05_Quality_Filtered")
fltr_low_quality_engine = MetaboIntFilter(data=inter_sc_data, pipeline_params=params)
low_quality_filter_data = fltr_low_quality_engine.execute_quality_filtering(output_dir=step5_dir)

2026-05-19 13:17:22.373 | INFO     | __main__:<module>:1 - Step 05: Low-Quality Feature Filtering...
2026-05-19 13:17:22.376 | INFO     | filtering:execute_quality_filtering:573 - Features before filtering: 347
2026-05-19 13:17:22.387 | INFO     | filtering:execute_quality_filtering:593 - Features after Blank/QC check: 268


2026-05-19 13:17:22.403 | INFO     | filtering:execute_quality_filtering:614 - Features after QC RSD check: 242
2026-05-19 13:17:22.414 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\05_Quality_Filtered.
2026-05-19 13:17:22.490 | INFO     | filtering:execute_quality_filtering:643 - Data after low-quality features filtering saved as: .\tutorial_output\05_Quality_Filtered\Filtered_Data_Low-quality_Features.csv


2026-05-19 13:17:23.663 | INFO     | filtering:_execute_s2_visualization:731 - Low-quality Filter summary dashboard saved as: .\tutorial_output\05_Quality_Filtered\Low-quality_Filtering_Dashboard.svg
2026-05-19 13:17:23.664 | SUCCESS  | filtering:execute_quality_filtering:654 - Low-quality features filtering completed.
2026-05-19 13:17:23.664 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_quality_filtering": 00:00:01.288.


### QA-Step 05: Quality Assessment on Low-Quality Feature Filtered Data
A pre-imputation health check is performed on the refined dataset. This evaluation confirms that the surviving features represent high-fidelity biological signals, ensuring the matrix is optimally prepared for missing value reconstruction

In [11]:
logger.info(
    "QA-Step 05: Quality Assessment on Low-Quality Feature Filtered Data...")

qa_step5_dir = os.path.join(OUTPUT_DIR, "QA_05_Quality_Filtered")
qa_low_quality_filter_engine = MetaboIntAssessor(
    data=low_quality_filter_data, pipeline_params=params)
qa_low_quality_filter_engine.execute_assessment(output_dir=qa_step5_dir)

2026-05-19 13:17:23.697 | INFO     | __main__:<module>:1 - QA-Step 05: Quality Assessment on Low-Quality Feature Filtered Data...
2026-05-19 13:17:23.698 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_05_Quality_Filtered.


2026-05-19 13:17:30.062 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_05_Quality_Filtered\QA_Summary_Dashboard.svg
2026-05-19 13:17:30.062 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:17:30.063 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.364.


## Step 06: Missing Value Imputation
An autonomous multi-algorithm benchmarking simulation is executed for MAR. The optimal algorithm is programmatically selected based on its ability to reconstruct established distributions while minimizing bias in the original variance structure.

Execute min-value constant imputation for MNAR.

In [12]:
logger.info("Step 06: Missing Value Imputation...")

step6_dir = os.path.join(OUTPUT_DIR, "06_Imputation")
imp_engine = MetaboIntImputer(data=low_quality_filter_data, pipeline_params=params)
imputed_data = imp_engine.execute_imputation(output_dir=step6_dir)

2026-05-19 13:17:30.105 | INFO     | __main__:<module>:1 - Step 06: Missing Value Imputation...
2026-05-19 13:17:30.109 | INFO     | imputation:execute_imputation:533 - Hybrid Imputation Engine Initialized. MAR: Auto | MNAR: Row-wise (LOD=0.5x) | KNN: 5 | Sim: 0.05
2026-05-19 13:17:30.115 | INFO     | imputation:execute_imputation:547 - Applying Row-wise-wise LOD to 3 MNAR.
2026-05-19 13:17:30.339 | INFO     | imputation:select_best_algorithm:435 - Simulating "KNN" on MAR subset...
2026-05-19 13:17:30.377 | DEBUG    | pimqc:_safe_popen:142 - Permitted probe: powershell.exe -Command '(Get-CimInstance' -ClassName 'Win32_Processor).NumberOfCores'
2026-05-19 13:17:32.237 | INFO     | imputation:select_best_algorithm:435 - Simulating "Probabilistic" on MAR subset...
2026-05-19 13:17:32.852 | INFO     | imputation:select_best_algorithm:435 - Simulating "Median" on MAR subset...
2026-05-19 13:17:33.207 | INFO     | imputation:select_best_algorithm:446 - Optimal MAR algorithm selected: KNN
202

2026-05-19 13:17:37.569 | INFO     | imputation:execute_imputation:653 - Imputer summary dashboard saved as: .\tutorial_output\06_Imputation\Imputation_Dashboard_KNN.svg
2026-05-19 13:17:37.569 | SUCCESS  | imputation:execute_imputation:656 - Missing value imputation completed successfully.
2026-05-19 13:17:37.572 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_imputation": 00:00:07.465.


### QA-Step 06: Quality Assessment on Imputed Data
Evaluates the extent to which synthetic data points might introduce artificial clustering or distort natural biological correlations, with a specific focus on the stability of low-abundance signals.

In [13]:
logger.info("QA-Step 06: Quality Assessment of Imputated Data...")

qa_step6_dir = os.path.join(OUTPUT_DIR, "QA_06_Imputed_Data")
qa_imp_engine = MetaboIntAssessor(data=imputed_data, pipeline_params=params)
qa_imp_engine.execute_assessment(output_dir=qa_step6_dir)

2026-05-19 13:17:37.611 | INFO     | __main__:<module>:1 - QA-Step 06: Quality Assessment of Imputated Data...
2026-05-19 13:17:37.612 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_06_Imputed_Data.


2026-05-19 13:17:43.736 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_06_Imputed_Data\QA_Summary_Dashboard.svg
2026-05-19 13:17:43.737 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:17:43.738 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.125.


## Step 07: Data Normalization
Applies advanced normalization techniques (e.g., VSN, Quantile) to stabilize heteroscedastic variance and harmonize global intensity scales across all biological samples.

In [14]:
logger.info("Step 07: Data Normalization...")

step7_dir = os.path.join(OUTPUT_DIR, "07_Normalized_Data")
norm_engine = MetaboIntNormalizer(imputed_data, pipeline_params=params)
normalized_data = norm_engine.execute_normalization(
    output_dir=step7_dir)

2026-05-19 13:17:43.783 | INFO     | __main__:<module>:1 - Step 07: Data Normalization...
2026-05-19 13:17:43.784 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\07_Normalized_Data.
2026-05-19 13:17:43.786 | INFO     | normalization:execute_normalization:525 - Permanently dropping 24 Blank samples.
2026-05-19 13:17:43.787 | INFO     | normalization:execute_normalization:527 - Applying Normalization | Method: VSN | Log: False


2026-05-19 13:18:17.867 | INFO     | normalization:execute_normalization:549 - Calculating normalization-related metrics...
2026-05-19 13:18:31.073 | INFO     | normalization:execute_normalization:556 - Generating diagnostic plots for normalization...


2026-05-19 13:18:36.494 | INFO     | normalization:execute_normalization:566 - Normalization summary dashboard saved as: .\tutorial_output\07_Normalized_Data\Normalization_Dashboard_VSN.svg
2026-05-19 13:18:36.494 | SUCCESS  | normalization:execute_normalization:567 - Data normalization completed successfully.
2026-05-19 13:18:36.495 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_normalization": 00:00:52.711.


### QA-Step 07: Quality Assessment on Normalized Data
Monitors spatial distribution fidelity using Jensen-Shannon Divergence (JSD) and Wasserstein metrics, evaluating whether the normalization successfully mitigated systematic biases without obliterating genuine biological differences.

In [15]:
logger.info("QA-Step 07: Quality Assessment of Normalized Data...")

qa_step7_dir = os.path.join(OUTPUT_DIR, "QA_07_Norm_Data")
qa_norm_engine = MetaboIntAssessor(
    data=normalized_data, pipeline_params=params)
qa_norm_engine.execute_assessment(output_dir=qa_step7_dir)

2026-05-19 13:18:36.550 | INFO     | __main__:<module>:1 - QA-Step 07: Quality Assessment of Normalized Data...
2026-05-19 13:18:36.551 | WARNING  | io_utils:_check_dir_exists:368 - No such directory, creating a new directory:
	.\tutorial_output\QA_07_Norm_Data.


2026-05-19 13:18:42.724 | INFO     | assessment:execute_assessment:481 - Assessor summary dashboard saved as: .\tutorial_output\QA_07_Norm_Data\QA_Summary_Dashboard.svg
2026-05-19 13:18:42.724 | SUCCESS  | assessment:execute_assessment:482 - Data quality assessment completed.
2026-05-19 13:18:42.726 | SUCCESS  | io_utils:time_wrap:393 - Execution time of "execute_assessment": 00:00:06.174.


## Step 08: Sequential Audit Report Compilation
Assembles all intermediate vector graphics, mathematical metrics, and tracking logs to autonomously generate a comprehensive, human-readable HTML/PDF and Markdown audit report.

In [16]:
logger.info("Step 08: Sequential Audit Report Compilation...")

REPORT_DIR = "08_Report_Summary"

# 1. Process Visual Assets
visual_rep = ru.VisualAssetReporter(
    base_dir=OUTPUT_DIR
)
visual_rep.compile_assessor_report(
    report_folder=REPORT_DIR, is_multi_batch=is_multi_batch_flag)

# 2. Extract Metadata & Render Markdown (Sequential Mapping)
# 2.1 Define the consolidated object pool using sub-step keys
pipeline_metrics_objs = {
    "raw_dataset": raw_data.dataset_metrics,
    "high_mv_feature_filtering": mv_filter_data.mv_filtering_metrics,
    "intra_signal_correction": intra_sc_data.correction_metrics,
    "inter_signal_correction": inter_sc_data.correction_metrics,
    "low_quality_feature_filtering": 
        low_quality_filter_data.quality_filtering_metrics,
    "missing_value_imputation": imputed_data.imputation_metrics,
    "normalization": normalized_data.normalization_metrics
}

qa_metrics_objs = {
    "raw_dataset": qa_raw_engine.assessment_metrics,
    "high_mv_feature_filtering": qa_mv_filter_engine.assessment_metrics,
    "intra_signal_correction": qa_intra_engine.assessment_metrics,
    "inter_signal_correction": qa_inter_engine.assessment_metrics,
    "low_quality_feature_filtering": 
        qa_low_quality_filter_engine.assessment_metrics, 
    "missing_value_imputation": qa_imp_engine.assessment_metrics,
    "normalization": qa_norm_engine.assessment_metrics
}

# 1.2 Initialize reporter and generate ONE markdown file
print(f"Initializing narrative reporter at workspace: {OUTPUT_DIR}")
md_reporter = ru.NarrativeStatsReporter(base_dir=OUTPUT_DIR)

md_reporter.generate_markdown(
    pipeline_metrics=pipeline_metrics_objs, 
    qa_metrics=qa_metrics_objs,
    report_folder=REPORT_DIR
)

success = md_reporter.export_report(pdf_engine="weasyprint")

if success:
    logger.success("PI-METABOQC PIPELINE COMPLETED SUCCESSFULLY.")

2026-05-19 13:18:42.776 | INFO     | __main__:<module>:1 - Step 08: Sequential Audit Report Compilation...
2026-05-19 13:18:42.778 | INFO     | report_utils:compile_assessor_report:231 - Multi-batch design detected. Assembling Batch Grid.


2026-05-19 13:18:42.878 | SUCCESS  | report_utils:compile_assessor_report:264 - Report SVG assets compiled at: tutorial_output\08_Report_Summary\assets
Initializing narrative reporter at workspace: .\tutorial_output
2026-05-19 13:18:43.053 | INFO     | report_utils:generate_markdown:1017 - Generating COMPREHENSIVE narrative report...
2026-05-19 13:18:43.106 | SUCCESS  | report_utils:generate_markdown:1027 - Comprehensive report generated: tutorial_output\08_Report_Summary\Report_Comprehensive.md
2026-05-19 13:18:43.106 | INFO     | report_utils:generate_markdown:1017 - Generating BRIEF narrative report...
2026-05-19 13:18:43.121 | SUCCESS  | report_utils:generate_markdown:1027 - Brief report generated: tutorial_output\08_Report_Summary\Report_Brief.md
2026-05-19 13:18:43.125 | DEBUG    | pimqc:_safe_popen:142 - Permitted probe: pandoc --version
2026-05-19 13:18:43.286 | INFO     | report_utils:export_report:1171 - --- Exporting PDF for: Report_Comprehensive.md ---
2026-05-19 13:18:43.2

In [17]:
print(iu.dir_tree(dir_path=OUTPUT_DIR))


tutorial_output
├── 01_Raw_Data
│   ├── Global_Acquisition_Overview.svg
│   └── Raw_Data_Intensity.csv
├── 02_MV_Filtered
│   ├── Filtered_Data_High-MV_Features.csv
│   ├── Filtered_Data_High-MV_Samples.csv
│   ├── Filtering_Tracking_High-MV_Features.csv
│   ├── Filtering_Tracking_High-MV_Samples.csv
│   └── MV_Classification_Dashboard.svg
├── 03_04_Corrected_Data
│   ├── Internal_Standard_Scatters
│   │   ├── IS_Scatter_Carnitine_C10_0_d3.svg
│   │   ├── IS_Scatter_CA_d4.svg
│   │   ├── IS_Scatter_CDCA_d4.svg
│   │   ├── IS_Scatter_Choline_d4.svg
│   │   └── IS_Scatter_Phenylalanine_d5.svg
│   ├── Inter_Batch_Corrected_QC-SVR.csv
│   ├── Intra_Batch_Corrected_QC-SVR.csv
│   ├── Pred_Base_IS_QC-SVR.svg
│   ├── QC_Fit_Baseline_QC-SVR.csv
│   └── QC_RSD_Boxplot_QC-SVR.svg
├── 05_Quality_Filtered
│   ├── Filtered_Data_Low-quality_Features.csv
│   ├── Filtering_Tracking_Low-quality_Features.csv
│   └── Low-quality_Filtering_Dashboard.svg
├── 06_Imputation
│   ├── Imputation_Dashboard_KNN.

In [18]:
md_reporter.consolidate_metrics(pipeline_metrics=pipeline_metrics_objs, 
    qa_metrics=qa_metrics_objs)

{'metadata': {'date': '2026-05-19 13:18',
  'mode': 'POS',
  'is_multi_batch': True},
 'raw_dataset': {'pipeline_params': {'mode': 'POS',
   'pi-metaboqc_version': '1.0.3a1',
   'features': {'total': 376,
    'internal_standards': ['CA-d4',
     'CDCA-d4',
     'Choline-d4',
     'Phenylalanine-d5',
     'Carnitine C10:0-d3'],
    'internal_standards_count': 5},
   'samples': {'total': 466, 'qc': 44, 'blank': 24, 'actual': 398},
   'batches': {'batch_count': 3,
    'ordered_batches': ['B1', 'B2', 'B3'],
    'batch_distribution': {'B1': {'Total': 154,
      'QC': 14,
      'Blank': 8,
      'Sample': 132,
      'Inject Order': '1 ~ 169'},
     'B2': {'Total': 155,
      'QC': 14,
      'Blank': 8,
      'Sample': 133,
      'Inject Order': '170 ~ 338'},
     'B3': {'Total': 157,
      'QC': 16,
      'Blank': 8,
      'Sample': 133,
      'Inject Order': '339 ~ 541'}}}},
  'qa_assessments': {'correlation': {'method': 'Spearman',
    'sample_level': {'inner_batch_median': 0.9707447127466